# L2R adjustments
Nic Tarasewicz
02/10/2024

Adjustments made to L2R (Webster et al., 2020). Follow installation and setup of CanRad (L2R) as described https://github.com/c-webster/CanRad.jl/tree/main.


Changes:

1. Use ```l2r_preprocess_lidar.R``` instead of ```canrad_pecalc.jl```. Could not get PyCrown installed/functional on macOS Sonoma 14.5 (Apple M1 Pro chop, 32GB RAM). 

2. Pay close attention to final lidar formatting. Segemented TreeID # must match intesnity value. 

3. Lidar *.laz must be in version 1.3 and point type 3 to load properly in L2R. Use LASTools to adjust: ```las2las64 -i "input_path.laz" -o "output_path.laz" -set_version 1.3 -set_point_type 3```

4. Use ```l2r_preprocess_radiation.ipynb``` for interpolating SWin observations for L2R input to 2-min and correct datetime format. 

5. In ```fileio.jl```, adjust ```ang = rand(Uniform(60,100),size(lastc,1))``` in ```loadltc_laz``` function to customize branching angle. 

6. In ```las2rad.jl```, change "SWR calcaulations" to:
    ```
    if calc_swr > 0
        radiation = RADIATION(loc_time = loc_time, swr_open = Vector{Float64}())       
        if calc_swr == 2
            swrdat   = readdlm(swrf)
            new_swr_open = float.(swrdat[:, 3]) # mod
            # Create a new instance of RADIATION with the updated swr_open
            # radiation = RADIATION(loc_time = radiation.loc_time, swr_open = new_swr_open)
            radiation = RADIATION(loc_time = loc_time, swr_open = new_swr_open)
        end
    end
    ```
    Use this ONLY for when loading SWin observation *.txt file ```"calc_swr" => 2"``` in ```L2R_settings_test.jl```. Keep unmodified version for potential swer (atm_trans=1)

7. Modify ```runtests.jl``` for local paths. Adjust ```@testset "selected_models" begin``` to: 
    ```
    @testset "selected_models" begin

        for model in ["T2R", "L2R"]  # Only include T2R and L2R

            setf       = joinpath(datdir, model * "_settings_test.jl")
            include(setf)
            pts = readdlm(ptsf)
            exdir      = joinpath(datdir, "Output_" * model)
            if model == "T2R"
                dat_in, par_in = ter2rad_settings(datdir)
                println("Running T2R ...")
                ter2rad!(pts, dat_in, par_in, exdir, "T2R test")

                ncds = Dataset(joinpath(exdir, "Output_testset.nc"))
                tst_dd = ncds["svf_planar_t"][:].*0.01
                close(ncds)

            elseif model == "L2R"
                dat_in, par_in = las2rad_settings(datdir)
                println("Testing L2R ...")
                las2rad!(pts, dat_in, par_in, exdir, "L2R test")

                ncds = Dataset(joinpath(exdir, "Output_testset.nc"))
                tst_dd = ncds["svf_planar"][:].*0.01
                close(ncds)
            end
        end
    end
    cd(cwd)
    ```
    
8. Adjust ```L2R_settings_test.jl``` for specific domain inputs, times, and runs. See accompanying files for specific run examples. 
    ```
    # terrain settings
            "terrain_highres"     => true, # include high high resolution local terrain (1-5 m)
            "highres_peri"        => 300, # radius for including high-res terrain !! should be float number!
            "terrain_lowres"      => true, # !MUST BE TRUE! include low resolution regional terrain (> 25 m)
            "lowres_peri"         => 10000, # radius around points to include lowres terrain
            "terrainmask_precalc" => true, # if the terrain mask has been pre-calculated by CanRad-T2R
            "calc_terrain"        => true, # calculate variables for terrain-only as well as forest
            "buildings"           => false, # include buildings as opague features (requires a building height model, bhmf, like the chm)
            "horizon_line"        => false, # use pre-calculated horizon line matrix (e.g. for compatibility with topo-downscaled swr data)

            # forest/lidar settings
            "keep_points"    => "ground+canopy", # "canopy", "ground+canopy", "all"
            "trunks"         => true, # true or false
            "branches"       => true, # true or false
            "branch_spacing" => 0.1, # spacing interval for creating branches if left empty the default is 0.1
            "season"         => "none", # "winter" or "nothing" see docs for more information about this feature
            "forest_peri"    => 100, # radius around points to include in image

            # calculation settings
            "calc_trans" => true,
            "calc_swr" => 2, # 0 = off; 1 = potential swr (atm_trans = 1); 2 = real swr (needs swrf in dat_in)
    ...
    "image_height" => 2.0,
    "point_size" => [30.0,10.0],
    ```